# 🤖 Pocket OTC AI Analyzer — Telegram + Historical Patterns

شغّل الخلية الوحيدة أدناه. تقوم بتنزيل آخر نسخة، تثبيت المتطلبات، محاولة جلب عينات OTC التاريخية من مستودع البحث، التحقق من Telegram وOpenRouter، ثم تشغيل البوت.

المحرك الجديد لا يعتمد على EMA/RSI لإجبار إشارة؛ يبحث عن حالات سعرية تاريخية مشابهة، وإذا لم يجد عينة كافية يعيد NO TRADE.

READ-ONLY: البوت لا يسجل الدخول إلى Pocket Option ولا ينفذ صفقات.

In [ ]:
# 🚀 شغّل هذه الخلية فقط
import os,sys,subprocess,shutil,getpass,traceback,signal,time
REPO='https://github.com/mohmb142/Jjjjjjj.git'; ROOT='/content/Jjjjjjj'
DATA_REPO='https://github.com/PHCStanton/pocket-option-otc-dataset.git'; DATA_ROOT='/content/otc-source'
try:
 os.chdir('/content')
 old=subprocess.run(['bash','-lc',"ps -eo pid,args | grep '[t]elegram_bot.py' | awk '{print $1}'"],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
 for x in old.stdout.splitlines():
  try: os.kill(int(x.strip()),signal.SIGTERM)
  except: pass
 time.sleep(1)
 if os.path.exists(ROOT): shutil.rmtree(ROOT,ignore_errors=True)
 print('1/5 ⬇️ تنزيل آخر نسخة من المشروع...')
 p=subprocess.run(['git','clone','--depth','1',REPO,ROOT],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,timeout=180)
 print(p.stdout[-3000:]); assert p.returncode==0,'فشل تنزيل GitHub'
 os.chdir(ROOT)
 print('2/5 📦 تثبيت المتطلبات...')
 p=subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,timeout=600)
 print(p.stdout[-5000:]); assert p.returncode==0,'فشل تثبيت المتطلبات'
 print('3/5 📚 محاولة جلب عينات OTC التاريخية (اختياري)...')
 try:
  if os.path.exists(DATA_ROOT): shutil.rmtree(DATA_ROOT,ignore_errors=True)
  p=subprocess.run(['git','clone','--depth','1','--filter=blob:none','--sparse',DATA_REPO,DATA_ROOT],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,timeout=240)
  if p.returncode==0:
   subprocess.run(['git','-C',DATA_ROOT,'sparse-checkout','set','data/samples'],stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,timeout=60)
   os.environ['PATTERN_DATASET_DIR']=DATA_ROOT+'/data/samples'
   print('✅ تم تجهيز عينات OTC في:',os.environ['PATTERN_DATASET_DIR'])
  else: print('⚠️ تعذر جلب العينات؛ سيستخدم المحرك الشموع التي يجلبها API.')
 except Exception as e: print('⚠️ تخطي Dataset:',e)
 print('4/5 🔐 التحقق من المفاتيح وTelegram...')
 token=getpass.getpass('🤖 Telegram Bot Token: ').strip(); key=getpass.getpass('🧠 OpenRouter API Key: ').strip()
 if not token or not key: raise RuntimeError('يجب إدخال المفتاحين')
 os.environ['TELEGRAM_BOT_TOKEN']=token; os.environ['OPENROUTER_API_KEY']=key; os.environ.setdefault('OPENROUTER_MODEL','google/gemini-2.5-flash')
 check="import asyncio,os\nfrom telegram import Bot\nasync def t():\n b=Bot(os.environ['TELEGRAM_BOT_TOKEN']); m=await b.get_me(); print(f'CONNECTED: @{m.username} | id={m.id}'); await b.delete_webhook(drop_pending_updates=False)\nasyncio.run(t())"
 p=subprocess.run([sys.executable,'-c',check],cwd=ROOT,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,timeout=60)
 print(p.stdout); assert p.returncode==0,'فشل اتصال Telegram: تحقق من Token أو الشبكة'
 print('5/5 🤖 تشغيل البوت الآن...')
 print('✅ Telegram متصل. أرسل /start ثم صورة الشارت.')
 subprocess.run([sys.executable,'telegram_bot.py'],cwd=ROOT,check=True)
except KeyboardInterrupt: print('\n🛑 تم إيقاف البوت.')
except Exception: print('\n❌ حدث خطأ:'); traceback.print_exc()
